# Retos C3, C4 y C5 · Fabiola Valencia Barrios
Esta copia reúne el entorno completo de 03 para ejecutar los retos sin depender de variables externas. Se conservan las mismas celdas y salidas de la ejecución original de 03; no son una segunda medición independiente.
Las celdas finales implementan evaluación hit@k, MMR y Gradio con filtro por metadata. Los resultados y límites se analizan en RESPUESTAS.md.


<a href="https://colab.research.google.com/gist/Fabi-22/f30dd01e68ae74f52068e6b0150162bb/copia-de-03_rag_workshop_papers_nvidia_nim_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/llama-nemotron-embed-vl-1b-v2`
- Generación: `nvidia/nemotron-3.5-super-vl-120b-a12b`


In [1]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 21.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take i

In [6]:
import os
import getpass
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY").strip()
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2"

print("✅ Configuración NVIDIA NIM lista")


✅ Configuración NVIDIA NIM lista


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


In [4]:
from markitdown import MarkItDown

DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return

    response = requests.get(url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")


for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")


⬇ Descargado: attention_paper.pdf (2.2 MB)
⬇ Descargado: rag_paper.pdf (0.9 MB)
✅ attention_paper.pdf → 40,174 caracteres
✅ rag_paper.pdf       → 69,121 caracteres

Inicio del paper de Attention:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog...


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)


Attention → 55 chunks
RAG       → 95 chunks
Total     → 150 chunks

Ejemplo de metadatos: {'title': 'Attention Is All You Need', 'source': 'attention_paper.pdf', 'chunk_index': 0}


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [7]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    truncate="END",
)
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = vectorstore.add_documents(all_docs)

print(f"✅ {len(ids)} chunks indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/llama-nemotron-embed-vl-1b-v2 in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ 150 chunks indexados en Qdrant
✅ Dimensión de embeddings: 2048


---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [15]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=800,
).bind(chat_template_kwargs={"enable_thinking":False}).with_retry(stop_after_attempt=4)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts. "
        "If the excerpts do not contain enough evidence, say so explicitly. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = rag_chain.invoke(test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)


Pregunta: What is the Transformer architecture?
Respuesta: The Transformer is a sequence transduction model based entirely on attention mechanisms, dispensing with recurrence and convolutions entirely. It follows an encoder-decoder structure: the encoder maps an input sequence of symbol representations to a sequence of continuous representations, and the decoder generates an output sequence one element at a time in an auto-regressive fashion. Both the encoder and decoder are composed of stacked identical layers. Each encoder layer has two sub-layers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network, with residual connections and layer normalization around each sub-layer. All sub-layers and embedding layers produce outputs of dimension d_model = 512. (Attention Is All You Need)


---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [16]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

Q: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?
A: The Transformer (big model) achieved a BLEU score of 28.4 on WMT 2014 English-to-German. (Attention Is All You Need)


<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [17]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

Q: How does RAG combine parametric and non-parametric memory?
A: RAG combines parametric and non-parametric memory by using a pre-trained seq2seq transformer (BART) as parametric memory and a dense vector index of Wikipedia accessed by a pre-trained neural retriever (DPR) as non-parametric memory. These components are combined in a probabilistic model trained end-to-end: the retriever provides latent documents conditioned on the input, and the seq2seq model conditions on these latent documents together with the input to generate the output. Latent documents are marginalized with a top-K approximation, either per-output (same document for all tokens) or per-token (different documents for different tokens). (Retrieval-Augmented Generation | rag_paper.pdf)


<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [18]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

Q: What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?
A: The scaling factor √dk counteracts large dot products for large dk, which otherwise push the softmax into regions with extremely small gradients. (Attention Is All You Need)


<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [19]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

Q: What are the two RAG model variants proposed in the paper and how do they differ?
A: The two variants are RAG-Sequence and RAG-Token. RAG-Sequence uses the same retrieved document to generate each target token, treating the document as a single latent variable marginalized over top-K documents. RAG-Token can predict each target token based on a different document, allowing the generator to choose content from several documents when producing an answer. (Retrieval-Augmented Generation | rag_paper.pdf)


<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [20]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

Q: How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?
A: The base Transformer model uses 8 attention heads, with key dimension d_k = 64 and value dimension d_v = 64 per head. (Attention Is All You Need)


<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [21]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = rag_chain.invoke(q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

Q: The RAG paper uses FAISS for fast retrieval with an HNSW approximation. The Attention paper proposes the Transformer architecture. What seq2seq model does RAG use as its generator component, and is it based on the Transformer?

A: The RAG paper uses BART as its generator component, and yes, it is based on the Transformer architecture. The paper states that the generator is "a pre-trained BART model" and describes BART as "a denoising autoencoder for pretraining sequence-to-sequence models" based on the Transformer architecture. (Retrieval-Augmented Generation | rag_paper.pdf)

Fuentes recuperadas:
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf


<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = [
    (attention_text, "Attention Is All You Need", "attention_paper.pdf"),
    (rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf"),
]

def build_store(chunk_size: int, chunk_overlap: int):
    """Chunking + embeddings + Qdrant en memoria. Devuelve (store, n_chunks)."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    docs = []
    for text, title, source in CORPUS:
        for i, chunk in enumerate(splitter.split_text(text)):
            if len(chunk.strip()) >= 30:
                docs.append(Document(page_content=chunk,
                                     metadata={"title": title, "source": source, "chunk_index": i}))
    name = f"exp_{chunk_size}_{chunk_overlap}"
    client = QdrantClient(":memory:")
    client.create_collection(collection_name=name,
                             vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
    store = QdrantVectorStore(client=client, collection_name=name, embedding=embeddings)
    store.add_documents(docs)
    return store, len(docs)


def build_chain(store, k: int = 5, threshold: float = 0.25):
    """Reutiliza un índice ya construido y cambia solo la recuperación."""
    retr = store.as_retriever(search_type="similarity_score_threshold",
                              search_kwargs={"k": k, "score_threshold": threshold})
    chain = ({"context": retr | format_docs, "question": RunnablePassthrough()}
             | prompt | llm | StrOutputParser())
    return chain, retr


# B1–B5 y retos C4/C5: resultados completos, sin claves.
import json, time, csv, re
from pathlib import Path
B_RESULTS={'model':GENERATION_MODEL,'embedding_model':EMBEDDING_MODEL,'baseline_chunks':len(all_docs)}

def save_b():
    Path('resultados_B.json').write_text(json.dumps(B_RESULTS,ensure_ascii=False,indent=2),encoding='utf-8')

def safe_invoke(chain,question):
    for attempt in range(4):
        try:return chain.invoke(question)
        except Exception as error:
            if attempt==3:raise
            print('Reintento NVIDIA',attempt+1,type(error).__name__,flush=True)
            time.sleep(2**attempt)

def row_for(chain,retr,question,answer=None):
    docs=retr.invoke(question)
    if answer is None:answer=safe_invoke(chain,question)
    return {'question':question,'answer':answer,'metadata':[d.metadata for d in docs],'contexts':[d.page_content for d in docs]}

QUIZ=[q1,q2,q3,q4,q5]
B_RESULTS['B1']=[row_for(rag_chain,retriever,q,a) for q,a in zip(QUIZ+[q_bonus],[answer1,answer2,answer3,answer4,answer5,answer_bonus])]
save_b(); print('B1 registrado',flush=True)
# Se reutiliza el índice base de 900/120 ya construido: total tres índices.
STORES={900:(vectorstore,len(all_docs))}
for size,overlap in [(400,52),(1500,195)]:
    STORES[size]=build_store(size,overlap)
    print('Índice',size,STORES[size][1],'chunks',flush=True)
B_RESULTS['B2']=[]
configs=[('chunk_size',400,52,5,0.25),('chunk_size',900,120,5,0.25),('chunk_size',1500,195,5,0.25),('k',900,120,3,0.25),('k',900,120,5,0.25),('k',900,120,8,0.25),('score_threshold',900,120,5,0.15),('score_threshold',900,120,5,0.25),('score_threshold',900,120,5,0.40)]
for param,size,overlap,k,threshold in configs:
    store,n=STORES[size]; chain,retr=build_chain(store,k=k,threshold=threshold)
    rows=[]
    for question in QUIZ:rows.append(row_for(chain,retr,question));time.sleep(0.5)
    result={'parameter':param,'chunk_size':size,'overlap':overlap,'k':k,'threshold':threshold,'n_chunks':n,'rows':rows}
    B_RESULTS['B2'].append(result);save_b();print('B2',param,size,k,threshold,'completo',flush=True)
# Preguntas plausibles sobre datos posteriores o ausentes.
TRAPS=[
'What BLEU score did GPT-5 achieve when the authors compared it with the Transformer in this paper?',
'How much electricity in kWh did the authors measure for training the base Transformer?',
'What exact benchmark accuracy did the RAG authors report for Llama 4 in 2025?']
B_RESULTS['B3']=[row_for(rag_chain,retriever,q) for q in TRAPS];save_b();print('B3 completo',flush=True)
# Se conserva el prompt original si todas las respuestas admiten falta de evidencia;
# se evalúa y documenta la clasificación en RESPUESTAS.md tras leer las salidas.
ES=[
'¿Qué puntuación BLEU obtuvo el Transformer grande en WMT 2014 de inglés a alemán?',
'¿Para qué sirve el factor de escala raíz cuadrada de d_k en la atención por producto escalar escalado?',
'¿Cuáles son las dos variantes de RAG propuestas en el artículo y en qué se diferencian?']
B_RESULTS['B4']=[{'english':B_RESULTS['B1'][i],'spanish':row_for(rag_chain,retriever,q)} for i,q in zip([0,2,3],ES)];save_b();print('B4 completo',flush=True)
NEW_QUIZ=[
{'question':'How many identical layers are used in the encoder stack of the base Transformer?', 'ground_truth':'The encoder contains N=6 identical layers.','section':'Attention §3.1 Encoder and Decoder Stacks','source':'attention_paper.pdf'},
{'question':'What non-parametric knowledge source is used by RAG, and how are Wikipedia articles split into passages?', 'ground_truth':'The non-parametric memory uses the December 2018 English Wikipedia dump; articles are split into disjoint 100-word passages, yielding about 21 million documents.','section':'RAG §3 Experimental Setup — Non-parametric Memory','source':'rag_paper.pdf'},
{'question':'Both papers use a Transformer-based sequence-to-sequence generator. What does Attention remove from its architecture, and what external memory does RAG add to its generator?', 'ground_truth':'Attention removes recurrence and convolutions, relying on attention. RAG uses BART-large as its seq2seq generator and augments it with a dense vector index of Wikipedia passages accessed by a retriever.','section':'Attention Abstract/§3; RAG §2.1 Retriever and §2.2 Generator','source':'ambos'}]
B_RESULTS['B5']=[]
for item in NEW_QUIZ:B_RESULTS['B5'].append({**item,**row_for(rag_chain,retriever,item['question'])})
save_b();print('B5 completo',flush=True)
# C4: diez preguntas con fuente y palabra clave verificable.
EVAL_ROWS=[
{'pregunta':q1,'fuente_esperada':'attention_paper.pdf','palabra_clave':'28.4'},
{'pregunta':q2,'fuente_esperada':'rag_paper.pdf','palabra_clave':'parametric'},
{'pregunta':q3,'fuente_esperada':'attention_paper.pdf','palabra_clave':'softmax'},
{'pregunta':q4,'fuente_esperada':'rag_paper.pdf','palabra_clave':'RAG-Token'},
{'pregunta':q5,'fuente_esperada':'attention_paper.pdf','palabra_clave':'64'},
{'pregunta':'How many identical layers are in the Transformer encoder stack?','fuente_esperada':'attention_paper.pdf','palabra_clave':'N = 6'},
{'pregunta':'How does the Transformer encode token positions?','fuente_esperada':'attention_paper.pdf','palabra_clave':'sinusoidal'},
{'pregunta':'Which pretrained seq2seq model is used as the RAG generator?','fuente_esperada':'rag_paper.pdf','palabra_clave':'BART'},
{'pregunta':'What size are the Wikipedia passages used by RAG?','fuente_esperada':'rag_paper.pdf','palabra_clave':'100'},
{'pregunta':'Which optimizer is used to train the Transformer?','fuente_esperada':'attention_paper.pdf','palabra_clave':'Adam'}]
Path('eval').mkdir(exist_ok=True)
with Path('eval/preguntas.csv').open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=['pregunta','fuente_esperada','palabra_clave']);writer.writeheader();writer.writerows(EVAL_ROWS)

def normalize(text):return re.sub(r'\s+','',text).casefold()
def hit_at_k(retriever,filas):
    hits=[]
    for fila in filas:
        docs=retriever.invoke(fila['pregunta'])
        hits.append(any(d.metadata.get('source')==fila['fuente_esperada'] and normalize(fila['palabra_clave']) in normalize(d.page_content) for d in docs))
    return 100*sum(hits)/len(hits)
B_RESULTS['C4']=[]
for size,(store,n) in STORES.items():
    for k in [3,5]:
        retr=store.as_retriever(search_kwargs={'k':k})
        B_RESULTS['C4'].append({'chunk_size':size,'k':k,'n_chunks':n,'hit_percent':hit_at_k(retr,EVAL_ROWS)})
B_RESULTS['eval_questions']=EVAL_ROWS;save_b();print('C4 completo',flush=True)
# C5: misma pregunta, dos estrategias, sin nuevas indexaciones.
SIM=vectorstore.as_retriever(search_type='similarity',search_kwargs={'k':5})
MMR=vectorstore.as_retriever(search_type='mmr',search_kwargs={'k':5,'fetch_k':20,'lambda_mult':0.5})
B_RESULTS['C5']=[]
for question in [q1,q4,q_bonus]:
    sides={}
    for name,retr in [('similarity',SIM),('mmr',MMR)]:
        docs=retr.invoke(question);sides[name]=[{'metadata':d.metadata,'content':d.page_content} for d in docs]
    B_RESULTS['C5'].append({'question':question,**sides})
save_b();print('C5 completo',flush=True)
print('EXPORT_B='+json.dumps(B_RESULTS,ensure_ascii=True),flush=True)
print('✅ Experimentos B1–B5 y C4/C5 terminados',flush=True)


B1 registrado
Índice 400 326 chunks
Índice 1500 96 chunks
B2 chunk_size 400 5 0.25 completo
B2 chunk_size 900 5 0.25 completo
B2 chunk_size 1500 5 0.25 completo
B2 k 900 3 0.25 completo
B2 k 900 5 0.25 completo
B2 k 900 8 0.25 completo
B2 score_threshold 900 5 0.15 completo
B2 score_threshold 900 5 0.25 completo
B2 score_threshold 900 5 0.4 completo
B3 completo
B4 completo
B5 completo
C4 completo
C5 completo
EXPORT_B={"model": "nvidia/nemotron-3.5-super-vl-120b-a12b", "embedding_model": "nvidia/llama-nemotron-embed-vl-1b-v2", "baseline_chunks": 150, "B1": [{"question": "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?", "answer": "The Transformer (big model) achieved a BLEU score of 28.4 on WMT 2014 English-to-German. (Attention Is All You Need)", "metadata": [{"title": "Attention Is All You Need", "source": "attention_paper.pdf", "chunk_index": 28, "_id": "9e8ff21f978a4398ba95084d9e1f47d9", "_collection_name": "academic_papers_nvidia"}, {"title": 

In [23]:
# C3: interfaz de papers con filtro Qdrant por metadata.source.
import gradio as gr
from qdrant_client.models import Filter, FieldCondition, MatchValue
PAPER_CHOICES={'Todos':None,'Attention Is All You Need':'attention_paper.pdf','Retrieval-Augmented Generation':'rag_paper.pdf'}
def paper_answer(question,paper='Todos'):
    source=PAPER_CHOICES[paper]
    filt=Filter(must=[FieldCondition(key='metadata.source',match=MatchValue(value=source))]) if source else None
    scored=vectorstore.similarity_search_with_score(question,k=5,filter=filt)
    docs=[d for d,s in scored if s>=0.25]
    answer=safe_invoke(prompt|llm|StrOutputParser(),{'context':format_docs(docs),'question':question})
    sources='\n'.join(f"- **{d.metadata['title']}** · `{d.metadata['source']}` · chunk {d.metadata['chunk_index']} · score **{s:.4f}**" for d,s in scored)
    return answer,sources,[{'metadata':d.metadata,'score':float(s),'content':d.page_content} for d,s in scored]
def paper_chat(question,paper):
    answer,sources,_=paper_answer(question,paper)
    return answer,sources
COMPARE_Q='How many attention heads does the base Transformer model use?'
B_RESULTS['C3']=[]
for selection in ['Todos','Retrieval-Augmented Generation']:
    answer,sources,scored=paper_answer(COMPARE_Q,selection)
    B_RESULTS['C3'].append({'question':COMPARE_Q,'filter':selection,'answer':answer,'retrieved':scored})
save_b()
print('EXPORT_C3='+json.dumps(B_RESULTS['C3'],ensure_ascii=True),flush=True)
with gr.Blocks(title='RAG de papers · Fabiola Valencia Barrios') as demo_papers:
    gr.Markdown('# RAG de papers\nFabiola Valencia Barrios · SI3003\nNVIDIA NIM + LangChain + Qdrant')
    paper_select=gr.Dropdown(choices=list(PAPER_CHOICES),value='Todos',label='Paper')
    question=gr.Textbox(label='Pregunta',value=COMPARE_Q)
    send=gr.Button('Consultar',variant='primary')
    answer_box=gr.Textbox(label='Respuesta',lines=7)
    sources_box=gr.Markdown()
    send.click(paper_chat,[question,paper_select],[answer_box,sources_box])
    question.submit(paper_chat,[question,paper_select],[answer_box,sources_box])
demo_papers.queue().launch(share=True,debug=False,show_error=True)


EXPORT_C3=[{"question": "How many attention heads does the base Transformer model use?", "filter": "Todos", "answer": "The base Transformer model uses 8 attention heads. This is stated in the section describing multi-head attention: \"In this work we employ h = 8 parallel attention layers, or heads,\" and confirmed in Table 3 where the base model lists h = 8. [Attention Is All You Need | attention_paper.pdf]", "retrieved": [{"metadata": {"title": "Attention Is All You Need", "source": "attention_paper.pdf", "chunk_index": 14, "_id": "8f2fab3f79fc40819924e1ad269f6343", "_collection_name": "academic_papers_nvidia"}, "score": 0.42254277356043235, "content": "output values. These are concatenated and once again projected, resulting in the final values, as\ndepictedinFigure2.\nMulti-headattentionallowsthemodeltojointlyattendtoinformationfromdifferentrepresentation\nsubspacesatdifferentpositions. Withasingleattentionhead,averaginginhibitsthis.\nMultiHead(Q,K,V)=Concat(head ,...,head )WO\n1 h

In [24]:
B_RESULTS['B5'][1]['section'] = 'RAG §3 Experiments, primer párrafo'
B_RESULTS['B5'][2]['section'] = 'Attention Abstract/§3; RAG §2.2 Retriever: DPR y §2.3 Generator: BART'
save_b()
print('EXPORT_B_FINAL=' + json.dumps(B_RESULTS, ensure_ascii=True))
print('URL_PAPERS=' + str(demo_papers.share_url))

EXPORT_B_FINAL={"model": "nvidia/nemotron-3.5-super-vl-120b-a12b", "embedding_model": "nvidia/llama-nemotron-embed-vl-1b-v2", "baseline_chunks": 150, "B1": [{"question": "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?", "answer": "The Transformer (big model) achieved a BLEU score of 28.4 on WMT 2014 English-to-German. (Attention Is All You Need)", "metadata": [{"title": "Attention Is All You Need", "source": "attention_paper.pdf", "chunk_index": 28, "_id": "9e8ff21f978a4398ba95084d9e1f47d9", "_collection_name": "academic_papers_nvidia"}, {"title": "Attention Is All You Need", "source": "attention_paper.pdf", "chunk_index": 1, "_id": "e1a98a7c2fdf4ffd9341c342820632f5", "_collection_name": "academic_papers_nvidia"}, {"title": "Attention Is All You Need", "source": "attention_paper.pdf", "chunk_index": 26, "_id": "d9e8aaee9d7a491e81189e45f5dfe4d7", "_collection_name": "academic_papers_nvidia"}, {"title": "Attention Is All You Need", "source": "atten

# B5 · Quiz propio con ground truth verificado

Las tres preguntas se ejecutan en la celda de experimentos B5; sus respuestas y fragmentos quedan guardados en `B_RESULTS["B5"]`.

## Pregunta propia 1

How many identical layers are used in the encoder stack of the base Transformer?

<details>
<summary>Ground truth y sección del paper</summary>

The encoder contains N=6 identical layers.

**Sección:** Attention §3.1 Encoder and Decoder Stacks.

**Fuentes:** [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) y [Retrieval-Augmented Generation](https://arxiv.org/pdf/2005.11401), según la pregunta.

</details>

## Pregunta propia 2

What non-parametric knowledge source is used by RAG, and how are Wikipedia articles split into passages?

<details>
<summary>Ground truth y sección del paper</summary>

The non-parametric memory uses the December 2018 English Wikipedia dump; articles are split into disjoint 100-word passages, yielding about 21 million documents.

**Sección:** RAG §3 Experiments, primer párrafo.

**Fuentes:** [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) y [Retrieval-Augmented Generation](https://arxiv.org/pdf/2005.11401), según la pregunta.

</details>

## Pregunta propia 3

Both papers use a Transformer-based sequence-to-sequence generator. What does Attention remove from its architecture, and what external memory does RAG add to its generator?

<details>
<summary>Ground truth y sección del paper</summary>

Attention removes recurrence and convolutions, relying on attention. RAG uses BART-large as its seq2seq generator and augments it with a dense vector index of Wikipedia passages accessed by a retriever.

**Sección:** Attention Abstract/§3; RAG §2.2 Retriever: DPR y §2.3 Generator: BART.

**Fuentes:** [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) y [Retrieval-Augmented Generation](https://arxiv.org/pdf/2005.11401), según la pregunta.

</details>

La tercera pregunta exige combinar ambos papers. La respuesta obtenida fue parcial: omitió las convoluciones y no recuperó evidencia del paper de Attention; se registra esa limitación en RESPUESTAS.md.
